# for_loops
`for` loops are how Python moves through data predictably. In real systems, they power log scans, record transformation, validation passes, batch processing, and metrics aggregation. The key engineering question is not how to loop, but how to loop without hiding state changes or wasting work.

## 1. Problem First
Why does this matter in real systems?
- Pipelines transform one record at a time.
- Monitoring systems aggregate events from streams.
- A badly designed loop can create bugs, duplicate writes, or unnecessary latency.

In [1]:
levels = ["INFO", "ERROR", "WARN"]
for level in levels:
    print(level)

INFO
ERROR
WARN


## 2. Minimal Concept
Core syntax:
- `for item in iterable:`
- The loop runs once for each value produced by the iterable.
- Lists, tuples, strings, files, dictionaries, and generators are iterable.

## 3. Mental Model
How Python actually behaves
A `for` loop does not repeatedly index into a container by default. It asks an iterable for the next value until iteration ends. That means the loop is really a protocol-driven consumer of values, not just a counting construct.

In [2]:
text = "api"
for char in text:
    print(char)

numbers = [10, 20, 30]
for value in numbers:
    print(value)

a
p
i
10
20
30


## 4. Internal Mechanics
Python turns a `for` loop into iterator creation plus repeated `next()` calls until `StopIteration` ends the loop. That is why custom objects can participate in `for` loops by implementing the iteration protocol.

In [3]:
import dis

def print_items(items):
    for item in items:
        print(item)

dis.dis(print_items)

iterator = iter([1, 2])
print(next(iterator))
print(next(iterator))

  3           0 RESUME                   0

  4           2 LOAD_FAST                0 (items)
              4 GET_ITER
        >>    6 FOR_ITER                13 (to 36)
             10 STORE_FAST               1 (item)

  5          12 LOAD_GLOBAL              1 (NULL + print)
             22 LOAD_FAST                1 (item)
             24 CALL                     1
             32 POP_TOP
             34 JUMP_BACKWARD           15 (to 6)

  4     >>   36 END_FOR
             38 RETURN_CONST             0 (None)
1
2


## 5. Edge Cases
Where it breaks:
- Mutating a list while iterating over it can skip or duplicate work.
- Loop variables leak into the outer scope after the loop finishes.
- Side effects inside loops can make retries and testing harder.

In [4]:
items = [1, 2, 3, 4]
for item in items:
    if item % 2 == 0:
        items.remove(item)
print(items)

for last_value in [7, 8, 9]:
    pass
print(last_value)

[1, 3]
9


## 6. Performance Thinking
- A loop over `n` items is usually O(n).
- Expensive work inside the loop dominates performance far more than loop syntax.
- Repeated allocation or network calls inside loops are common bottlenecks.
- Sometimes a generator pipeline or vectorized approach is cleaner, but plain loops are often the most readable correct tool.

## 7. Real Use Case
A log processor scans events and counts only the records that need alerting.

In [5]:
records = [
    {"level": "INFO"},
    {"level": "ERROR"},
    {"level": "ERROR"},
]
error_count = 0
for record in records:
    if record["level"] == "ERROR":
        error_count += 1
print(error_count)

2


## 8. Anti-Pattern
What beginners do wrong:
- Mutate the collection they are iterating over.
- Put too much unrelated logic inside one loop body.
- Depend on hidden outer state instead of making loop inputs and outputs obvious.

In [6]:
users = ["alice", "", "bob"]
clean_users = []
for user in users:
    if user:
        clean_users.append(user.strip())
print(clean_users)

['alice', 'bob']


## 9. Interview Signals
Questions FAANG asks:
- How does a `for` loop work under the hood in Python?
- Why is mutating a list during iteration dangerous?
- When should you use a loop versus a comprehension?
- What determines the performance of a loop in real systems?

## 10. Exercise (Non-trivial)
You are building a batch validator for incoming API payloads. Iterate through the records, separate valid and invalid entries, collect error reasons, and make sure the loop structure stays readable even as validation rules grow.

In [7]:
def split_records(records):
    valid = []
    invalid = []
    for record in records:
        if record.get("id"):
            valid.append(record)
        else:
            invalid.append(record)
    return valid, invalid

# Task:
# 1. Avoid truthiness bugs for id=0 if that can be valid.
# 2. Capture error reasons, not just buckets.
# 3. Keep the loop easy to extend.